# CPU SfmLevenbergMarquardtOptimizer

> **Created by Codex.**

Use the public CPU structure-from-motion optimizer to choose landmark elimination independently from the linear solver.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

## Two independent choices

`SfmEliminationMode.Full` sends the joint system to the ordinary nonlinear-optimizer solver. `SfmEliminationMode.Schur` eliminates every active `Point3` and `Unit3` value, leaving poses, camera objects, calibrations, and all other value types in the reduced system. With `MULTIFRONTAL_SOLVER`, one cached factorization eliminates the landmarks, factors the reduced-system Schur complement, and back-substitutes through the same Bayes tree. Other solvers receive an explicitly materialized reduced graph. CPU defaults to Full, `MULTIFRONTAL_SOLVER`, and an automatically generated natural-landmark prefix followed by a METIS ordering of the reduced system. No optimizer template argument is needed, and a shared global calibration remains in the reduced system.

The linear-solver choices are the ordinary `NonlinearOptimizerParams.LinearSolverType` choices; there is no second SFM-specific CPU enum.

In [ ]:
import gtsam

params = gtsam.SfmLevenbergMarquardtParams.ceresDefaults()
params.setEliminationMode(gtsam.SfmEliminationMode.Full)
params.setLinearSolver(
    gtsam.NonlinearOptimizerParams.LinearSolverType.MULTIFRONTAL_SOLVER
)
print(params.getEliminationMode(), params.getLinearSolver())

## Ordering semantics

A Full-mode ordering contains every active variable, as it does for ordinary LM. A CPU Schur-mode parameter ordering contains every active key whose value is neither `Point3` nor `Unit3`, exactly once. `CreateReducedOrdering(graph, initial)` symbolically eliminates the natural `Point3`/`Unit3` ordering once and runs METIS on the reduced graph. `CreateSchurOrdering(graph, reduced_ordering)` prefixes the reduced ordering with those eliminated keys and returns the complete Schur ordering used by Full mode or the fused multifrontal path.

In [ ]:
# graph and initial are an SFM NonlinearFactorGraph and Values.
# reduced_ordering = (
#     gtsam.SfmLevenbergMarquardtOptimizer.CreateReducedOrdering(graph, initial)
# )
# point_first_ordering = (
#     gtsam.SfmLevenbergMarquardtOptimizer.CreateSchurOrdering(
#         graph, reduced_ordering
#     )
# )
# params.setOrdering(point_first_ordering)
# optimizer = gtsam.SfmLevenbergMarquardtOptimizer(graph, initial, params)
# result = optimizer.optimize()

## Solver details

For `Iterative`, attach `PCGSolverParameters` for PCG or `SubgraphSolverParameters` for SubgraphSolver. Missing or unknown iterative parameters produce the same error as ordinary nonlinear optimization.

`CHOLMOD` is available when GTSAM detects SuiteSparse CHOLMOD at configure time. It supports both modes, arbitrary variable dimensions, ordinary Jacobian and Hessian factors, user ordering, and symbolic reuse. Reduced Hessian blocks are loaded directly into CHOLMOD. A build without CHOLMOD throws an actionable error when it is selected.

For the fastest measured BAL path, use point-batched factors, Full mode, `MULTIFRONTAL_SOLVER`, and the complete ordering returned by `CreateSchurOrdering`. This is one point-first factorization with no reduced factor graph or second solve. Fused Schur mode performs the mathematically identical elimination internally but measured slightly slower. Selecting CHOLMOD, legacy Cholesky/QR, or an iterative solver with Schur mode requests explicit `Point3`/`Unit3` elimination followed by that solver on the reduced Hessian.